In [5]:
import pandas as pd

# ========== 1. 文件路径 ==========
files = {
    "allcond": "nrsa1314_allcond_05312019_0.csv",
    "bentcnts": "nrsa1314_bentcnts_04232019 (1).csv",
    "bentmet": "nrsa1314_bentmet_02132019 (1).csv",
    "bentmmi": "nrsa1314_bentmmi_04232019 (2).csv",
}

# ========== 2. 读取数据 ==========
dfs = {}
uid_sets = {}

for name, path in files.items():
    df = pd.read_csv(path)
    if "UID" not in df.columns:
        raise ValueError(f"{name} 文件中不存在 UID 列")
    dfs[name] = df
    uid_sets[name] = set(df["UID"].dropna())

# ========== 3. 每个文件唯一 UID 数量 ==========
print("===== 每个文件的唯一 UID 数量 =====")
for name, uids in uid_sets.items():
    print(f"{name}: {len(uids)}")

# ========== 4. 所有文件共有的 UID ==========
common_all = set.intersection(*uid_sets.values())

print("\n===== 五个文件共有的 UID =====")
print(f"五表共有 UID 数量: {len(common_all)}")

# ========== 5. 以 bentcnts 为 anchor 的可合并情况 ==========
bentcnts_uids = uid_sets["bentcnts"]

print("\n===== bentcnts 与其他文件的 UID 重合情况 =====")
for name, uids in uid_sets.items():
    if name == "bentcnts":
        continue
    overlap = bentcnts_uids & uids
    print(f"bentcnts ∩ {name}: {len(overlap)}")

# ========== 6. bentcnts 中“可被所有其他表联合”的 UID ==========
bentcnts_common_with_all = bentcnts_uids & common_all

print("\n===== bentcnts 中可与所有其他文件联合的 UID =====")
print(f"UID 数量: {len(bentcnts_common_with_all)}")

# ========== 7. 统计 bentcnts 中这些 UID 对应的“行数” ==========
bentcnts_df = dfs["bentcnts"]

rows_usable = bentcnts_df[
    bentcnts_df["UID"].isin(bentcnts_common_with_all)
]

print("\n===== bentcnts 中最终可用于多表合并的行数 =====")
print(f"行数（taxa-level rows）: {rows_usable.shape[0]}")

# ========== 8. 保存关键结果 ==========
pd.DataFrame({"UID": list(common_all)}).to_csv(
    "UID_common_all_5_tables.csv", index=False
)

pd.DataFrame({"UID": list(bentcnts_common_with_all)}).to_csv(
    "UID_bentcnts_anchor_usable.csv", index=False
)

rows_usable.to_csv(
    "bentcnts_rows_usable_for_merge.csv", index=False
)

print("\n===== 结果已保存为 CSV 文件 ✔ =====")

===== 每个文件的唯一 UID 数量 =====
allcond: 1853
bentcnts: 2255
bentmet: 2261
bentmmi: 2261

===== 五个文件共有的 UID =====
五表共有 UID 数量: 1848

===== bentcnts 与其他文件的 UID 重合情况 =====
bentcnts ∩ allcond: 1848
bentcnts ∩ bentmet: 2255
bentcnts ∩ bentmmi: 2255

===== bentcnts 中可与所有其他文件联合的 UID =====
UID 数量: 1848

===== bentcnts 中最终可用于多表合并的行数 =====
行数（taxa-level rows）: 78224

===== 结果已保存为 CSV 文件 ✔ =====


In [7]:
import pandas as pd

# ========== 1. 文件列表 ==========
files = {
    "allcond": "nrsa1314_allcond_05312019_0.csv",
    "bentcnts": "nrsa1314_bentcnts_04232019 (1).csv",
    "bentmet": "nrsa1314_bentmet_02132019 (1).csv",
    "bentmmi": "nrsa1314_bentmmi_04232019 (2).csv",

}

# ========== 2. 你关心的指标 ==========
indicators = [
    "PHYLUM",
    "CLASS",
    "ORDER",
    "FAMILY",
    "GENUS",
    "TARGET_TAXON",
    "TAXA_ID",
    "TOTAL300",
    "IS_DISTINCT300",
    "LAT_DD83",
    "LON_DD83",
    "AG_ECO9",
    "STRAH_CAT",
    "EPA_REG",
    "UID",
    "SITE_ID",
    "YEAR",
    "DATE_COL",
    "VISIT_NO",
    "SAMPLE_TYPE",
]

# ========== 3. 读取所有文件的列名 ==========
file_columns = {}

for name, path in files.items():
    df = pd.read_csv(path, nrows=0)  # 只读表头，速度快
    file_columns[name] = set(df.columns)

# ========== 4. 构建“指标在哪个文件里”的结果表 ==========
records = []

for indicator in indicators:
    presence = {
        "indicator": indicator
    }
    for file_name, cols in file_columns.items():
        presence[file_name] = indicator in cols
    records.append(presence)

result_df = pd.DataFrame(records)

# ========== 5. 打印结果 ==========
print("\n===== 指标在各文件中的存在情况（True = 存在）=====\n")
print(result_df)

# ========== 6. 保存结果 ==========
result_df.to_csv("indicator_file_presence_1314.csv", index=False)

print("\n结果已保存为 indicator_file_presence_1314.csv ✔")


===== 指标在各文件中的存在情况（True = 存在）=====

         indicator  allcond  bentcnts  bentmet  bentmmi
0           PHYLUM    False      True    False    False
1            CLASS    False      True    False    False
2            ORDER    False      True    False    False
3           FAMILY    False      True    False    False
4            GENUS    False      True    False    False
5     TARGET_TAXON    False      True    False    False
6          TAXA_ID    False      True    False    False
7         TOTAL300    False      True    False    False
8   IS_DISTINCT300    False      True    False    False
9         LAT_DD83     True     False    False     True
10        LON_DD83     True     False    False     True
11         AG_ECO9    False     False    False    False
12       STRAH_CAT     True     False    False    False
13         EPA_REG     True     False    False    False
14             UID     True      True     True     True
15         SITE_ID     True      True     True     True
16         

In [9]:
import pandas as pd

# ========== 1. 读取数据 ==========
bentcnts = pd.read_csv("nrsa1314_bentcnts_04232019 (1).csv")   # 主表，不去重
allcond  = pd.read_csv("nrsa1314_allcond_05312019_0.csv")
bentmet  = pd.read_csv("nrsa1314_bentmet_02132019 (1).csv")
bentmmi  = pd.read_csv("nrsa1314_bentmmi_04232019 (2).csv")

# ========== 2. 其他表按 UID 去重 ==========
allcond_uid = allcond.drop_duplicates(subset=["UID"])
bentmet_uid = bentmet.drop_duplicates(subset=["UID"])
bentmmi_uid = bentmmi.drop_duplicates(subset=["UID"])

# （可选）检查是否真的一 UID 一行
print("UID 行数检查：")
print("allcond:", allcond_uid.shape[0], "/", allcond["UID"].nunique())
print("bentmet:", bentmet_uid.shape[0], "/", bentmet["UID"].nunique())
print("bentmmi:", bentmmi_uid.shape[0], "/", bentmmi["UID"].nunique())

# ========== 3. 逐表 left merge ==========
merged = bentcnts.merge(
    allcond_uid,
    on="UID",
    how="left",
    suffixes=("", "_allcond")
)

merged = merged.merge(
    bentmet_uid,
    on="UID",
    how="left",
    suffixes=("", "_bentmet")
)

merged = merged.merge(
    bentmmi_uid,
    on="UID",
    how="left",
    suffixes=("", "_bentmmi")
)

# ========== 4. 合并后 sanity check ==========
print("\n===== 合并前后行数检查 =====")
print("bentcnts 行数:", bentcnts.shape[0])
print("merged 行数  :", merged.shape[0])

# ========== 5. 保存结果 ==========
merged.to_csv("nrsa1314_merged_bentcnts_anchor.csv", index=False)

print("\n合并完成 ✔ 输出文件：nrsa1314_merged_bentcnts_anchor.csv")

UID 行数检查：
allcond: 1853 / 1853
bentmet: 2261 / 2261
bentmmi: 2261 / 2261

===== 合并前后行数检查 =====
bentcnts 行数: 97114
merged 行数  : 97114

合并完成 ✔ 输出文件：nrsa1314_merged_bentcnts_anchor.csv


In [11]:
import pandas as pd

# ========== 1. 读取合并后的文件 ==========
input_file = "nrsa1314_merged_bentcnts_anchor.csv"
df = pd.read_csv(input_file)

# ========== 2. 需要保留的指标 ==========
columns_to_keep = [
    "UID",              # 放在第一列
    "PHYLUM",
    "CLASS",
    "ORDER",
    "FAMILY",
    "GENUS",
    "TARGET_TAXON",
    "TAXA_ID",
    "TOTAL300",
    "IS_DISTINCT300",
    "LAT_DD83",
    "LON_DD83",
    "AG_ECO9",
    "STRAH_CAT",
    "EPA_REG",
    "SITE_ID",
    "YEAR",
    "DATE_COL",
    "VISIT_NO",
    "SAMPLE_TYPE",
]

# ========== 3. 检查哪些列真实存在 ==========
existing_cols = [c for c in columns_to_keep if c in df.columns]
missing_cols = [c for c in columns_to_keep if c not in df.columns]

if missing_cols:
    print("⚠️ 以下列在数据中不存在，将被跳过：")
    for c in missing_cols:
        print(" -", c)

# ========== 4. 筛选并重排 ==========
filtered_df = df[existing_cols]

# ========== 5. 保存新文件 ==========
output_file = "nrsa1314_selected_indicators_uid_first.csv"
filtered_df.to_csv(output_file, index=False)

print("\n处理完成 ✔")
print("输出文件：", output_file)
print("最终列顺序：")
print(filtered_df.columns.tolist())

C:\Users\lzy65\AppData\Local\Temp\ipykernel_20696\522253991.py:5: DtypeWarning: Columns (32,45,80) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


⚠️ 以下列在数据中不存在，将被跳过：
 - AG_ECO9

处理完成 ✔
输出文件： nrsa1314_selected_indicators_uid_first.csv
最终列顺序：
['UID', 'PHYLUM', 'CLASS', 'ORDER', 'FAMILY', 'GENUS', 'TARGET_TAXON', 'TAXA_ID', 'TOTAL300', 'IS_DISTINCT300', 'LAT_DD83', 'LON_DD83', 'STRAH_CAT', 'EPA_REG', 'SITE_ID', 'YEAR', 'DATE_COL', 'VISIT_NO', 'SAMPLE_TYPE']


In [13]:
import pandas as pd

# ========== 1. 文件列表 ==========
files = {
    "allcond": "nrsa1314_allcond_05312019_0.csv",
    "bentcnts": "nrsa1314_bentcnts_04232019 (1).csv",
    "bentmet": "nrsa1314_bentmet_02132019 (1).csv",
    "bentmmi": "nrsa1314_bentmmi_04232019 (2).csv",

}

# ========== 2. 你关心的指标 ==========
indicators = [
    "PHYLUM",
    "CLASS",
    "ORDER",
    "FAMILY",
    "GENUS",
    "TARGET_TAXON",
    "TAXA_ID",
    "TOTAL300",
    "IS_DISTINCT300",
    "LAT_DD83",
    "LON_DD83",
    "AG_ECO9",
    "STRAH_CAT",
    "EPA_REG",
    "UID",
    "SITE_ID",
    "YEAR",
    "DATE_COL",
    "VISIT_NO",
    "SAMPLE_TYPE",
    "MMI_BENT",
    "BENT_MMI_COND",
    "OE_SCORE",
    "Nitrogen",
    "Phosphorus",
    "DO"
]

# ========== 3. 读取所有文件的列名 ==========
file_columns = {}

for name, path in files.items():
    df = pd.read_csv(path, nrows=0)  # 只读表头，速度快
    file_columns[name] = set(df.columns)

# ========== 4. 构建“指标在哪个文件里”的结果表 ==========
records = []

for indicator in indicators:
    presence = {
        "indicator": indicator
    }
    for file_name, cols in file_columns.items():
        presence[file_name] = indicator in cols
    records.append(presence)

result_df = pd.DataFrame(records)

# ========== 5. 打印结果 ==========
print("\n===== 指标在各文件中的存在情况（True = 存在）=====\n")
print(result_df)

# ========== 6. 保存结果 ==========
result_df.to_csv("indicator_file_presence_1314.csv", index=False)

print("\n结果已保存为 indicator_file_presence_1314.csv ✔")


===== 指标在各文件中的存在情况（True = 存在）=====

         indicator  allcond  bentcnts  bentmet  bentmmi
0           PHYLUM    False      True    False    False
1            CLASS    False      True    False    False
2            ORDER    False      True    False    False
3           FAMILY    False      True    False    False
4            GENUS    False      True    False    False
5     TARGET_TAXON    False      True    False    False
6          TAXA_ID    False      True    False    False
7         TOTAL300    False      True    False    False
8   IS_DISTINCT300    False      True    False    False
9         LAT_DD83     True     False    False     True
10        LON_DD83     True     False    False     True
11         AG_ECO9    False     False    False    False
12       STRAH_CAT     True     False    False    False
13         EPA_REG     True     False    False    False
14             UID     True      True     True     True
15         SITE_ID     True      True     True     True
16         

In [15]:
import pandas as pd

# ========== 1. 读取当前筛选后的文件 ==========
input_file = "step1.csv"
df = pd.read_csv(input_file)

# ========== 2. 需要追加到最后的指标 ==========
append_cols = [
    "MMI_BENT",
    "BENT_MMI_COND",
    "OE_SCORE",
]

# ========== 3. 判断哪些列真实存在 ==========
existing_append_cols = [c for c in append_cols if c in df.columns]
missing_append_cols = [c for c in append_cols if c not in df.columns]

if missing_append_cols:
    print("⚠️ 以下指标在数据中不存在，将不会加入：")
    for c in missing_append_cols:
        print(" -", c)

# ========== 4. 重新组织列顺序 ==========
# 先取原来的列（去掉可能已存在的 append 列，防止重复）
base_cols = [c for c in df.columns if c not in append_cols]

# 新的最终列顺序
final_cols = base_cols + existing_append_cols

final_df = df[final_cols]

# ========== 5. 保存新文件 ==========
output_file = "nrsa1314_selected_indicators_with_mmi.csv"
final_df.to_csv(output_file, index=False)

print("\n处理完成 ✔")
print("输出文件：", output_file)
print("最终列顺序：")
print(final_df.columns.tolist())

⚠️ 以下指标在数据中不存在，将不会加入：
 - MMI_BENT
 - BENT_MMI_COND
 - OE_SCORE

处理完成 ✔
输出文件： nrsa1314_selected_indicators_with_mmi.csv
最终列顺序：
['UID', 'PHYLUM', 'CLASS', 'ORDER', 'FAMILY', 'GENUS', 'TARGET_TAXON', 'TAXA_ID', 'TOTAL300', 'IS_DISTINCT300', 'LAT_DD83', 'LON_DD83', 'STRAH_CAT', 'EPA_REG', 'SITE_ID', 'YEAR', 'DATE_COL', 'VISIT_NO', 'SAMPLE_TYPE']


In [17]:
import pandas as pd

# ========== 1. 读取原始数据 ==========
bentcnts = pd.read_csv("nrsa1314_bentcnts_04232019 (1).csv")   # anchor
allcond  = pd.read_csv("nrsa1314_allcond_05312019_0.csv")
bentmet  = pd.read_csv("nrsa1314_bentmet_02132019 (1).csv")
bentmmi  = pd.read_csv("nrsa1314_bentmmi_04232019 (2).csv")

# ========== 2. 其他表按 UID 去重 ==========
allcond_uid = allcond.drop_duplicates(subset=["UID"])
bentmet_uid = bentmet.drop_duplicates(subset=["UID"])
bentmmi_uid = bentmmi.drop_duplicates(subset=["UID"])

# ========== 3. 合并（left merge，不改变 bentcnts 行数） ==========
merged = bentcnts.merge(allcond_uid, on="UID", how="left", suffixes=("", "_allcond"))
merged = merged.merge(bentmet_uid, on="UID", how="left", suffixes=("", "_bentmet"))
merged = merged.merge(bentmmi_uid, on="UID", how="left", suffixes=("", "_bentmmi"))

# ========== 4. 你最终要的列 ==========
base_columns = [
    "UID",              # 第一列
    "PHYLUM",
    "CLASS",
    "ORDER",
    "FAMILY",
    "GENUS",
    "TARGET_TAXON",
    "TAXA_ID",
    "TOTAL300",
    "IS_DISTINCT300",
    "LAT_DD83",
    "LON_DD83",
    "AG_ECO9",
    "STRAH_CAT",
    "EPA_REG",
    "SITE_ID",
    "YEAR",
    "DATE_COL",
    "VISIT_NO",
    "SAMPLE_TYPE",
]

# 追加到最后的 3 个指标
tail_columns = [
    "MMI_BENT",
    "BENT_MMI_COND",
    "OE_SCORE",
]

# ========== 5. 检查列是否存在 ==========
existing_base = [c for c in base_columns if c in merged.columns]
missing_base  = [c for c in base_columns if c not in merged.columns]

existing_tail = [c for c in tail_columns if c in merged.columns]
missing_tail  = [c for c in tail_columns if c not in merged.columns]

if missing_base:
    print("⚠️ 以下基础列不存在：")
    for c in missing_base:
        print(" -", c)

if missing_tail:
    print("⚠️ 以下 MMI / OE 指标不存在：")
    for c in missing_tail:
        print(" -", c)

# ========== 6. 重新排列并筛选 ==========
final_columns = existing_base + existing_tail
final_df = merged[final_columns]

# ========== 7. 保存结果 ==========
output_file = "nrsa1314_merged_selected_with_mmi.csv"
final_df.to_csv(output_file, index=False)

print("\n合并 + 筛选完成 ✔")
print("输出文件：", output_file)
print("最终列顺序：")
print(final_df.columns.tolist())

⚠️ 以下基础列不存在：
 - AG_ECO9

合并 + 筛选完成 ✔
输出文件： nrsa1314_merged_selected_with_mmi.csv
最终列顺序：
['UID', 'PHYLUM', 'CLASS', 'ORDER', 'FAMILY', 'GENUS', 'TARGET_TAXON', 'TAXA_ID', 'TOTAL300', 'IS_DISTINCT300', 'LAT_DD83', 'LON_DD83', 'STRAH_CAT', 'EPA_REG', 'SITE_ID', 'YEAR', 'DATE_COL', 'VISIT_NO', 'SAMPLE_TYPE', 'MMI_BENT', 'BENT_MMI_COND', 'OE_SCORE']
